# Step 06b — Filter edges by indirect-path support

Loads the graph annotated by step 06a (edge properties `indirectP3`, `indirectP5`), plots their distributions, and saves the LCC of the graph after keeping only edges with enough indirect support. The saved file can be passed to step 07.

In [ ]:
import time
import math
import random
import os
import heapq
import numpy as np
from graph_tool import Graph, GraphView
from graph_tool.draw import sfdp_layout, random_layout
from graph_tool.topology import label_components, label_largest_component
import graph_tool as gt
import matplotlib.pyplot as plt

print("graph_tool Version: ", print(gt.__version__))
print("OpenMP Enabled: ", gt.openmp_enabled())

if gt.openmp_enabled():
    print("Number of threads before: ", gt.openmp_get_num_threads())
    gt.openmp_set_num_threads(1)
    print("Number of threads after: ", gt.openmp_get_num_threads())

## Parameters
Set the project, sample and the suffix of the graph produced by step 06a.

In [ ]:
# Parameters
project_name = "260921Jac"
scratch_prefix = os.path.join(os.environ.get("SPAN_DATA_ROOT", "./orcd"), "scratch", project_name) + "/"

SAMPLE_ID = "M7"

# Sample radius; used only for the "nm per pixel" printout
total_radius = 2.5

###########
print("Project Name: ", project_name)
print("Sample ID: ", SAMPLE_ID)

## Load graph and basic statistics

In [ ]:
# Output of step 06a (06a_count_indirect_paths.py)
suffix = "postPipeline0302_Lev3_min_e2_min_d2_max_d30_LCC_indirectPaths"

filename = scratch_prefix + SAMPLE_ID + "_barcodesGraph_" + suffix + ".gt.gz"

print("Loading: ", filename)

t0 = time.perf_counter()
g = gt.load_graph(filename)
elapsed_min = (time.perf_counter() - t0) / 60.0

print(f"Loaded in {elapsed_min:.2f} minutes.")

In [ ]:
num_nodes = g.num_vertices()
num_edges = g.num_edges()

print(f"Number of nodes: {num_nodes:,}")
print(f"Number of edges: {num_edges:,}")

print(f"Directed: {g.is_directed()}")


print("nm per pixel: ", round(2 * np.sqrt(total_radius**2 * np.pi * 10**6 / num_nodes / np.pi), 2), "nm")

ep_w = g.edge_properties.get("weight", None)

total_w = int(np.sum(ep_w.a)) if ep_w is not None else g.num_edges()
print(f"Total edge weight: {total_w:,}")

## Distributions of indirect-path counts

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Extract edge-property arrays ---
if "indirectP3" not in g.ep or "indirectP5" not in g.ep:
    raise RuntimeError("Graph is missing edge properties 'indirectP3' and/or 'indirectP5'.")

p3 = g.ep["indirectP3"].a
p5 = g.ep["indirectP5"].a

# Keep only values > 0
p3_pos = p3[p3 > 0]
p5_pos = p5[p5 > 0]

n_edges_total = int(g.num_edges())

print(f"Total edges in g: {n_edges_total:,}")
print(f"indirectP3 > 0: {p3_pos.size:,}")
print(f"indirectP5 > 0: {p5_pos.size:,}")

# Shared integer bins so both series align
max_val = 0
if p3_pos.size:
    max_val = max(max_val, int(p3_pos.max()))
if p5_pos.size:
    max_val = max(max_val, int(p5_pos.max()))

if max_val == 0:
    raise RuntimeError("No positive values found in indirectP3 or indirectP5.")

xs = np.arange(1, max_val + 1)
bins = np.arange(0.5, max_val + 1.5, 1)

counts_p3, _ = np.histogram(p3_pos, bins=bins)
counts_p5, _ = np.histogram(p5_pos, bins=bins)

bar_w = 0.4

# ---------------- Plot 1: raw counts, side-by-side bars ----------------
plt.figure(figsize=(12, 7))
plt.bar(xs - bar_w/2, counts_p3, width=bar_w, alpha=0.85, label="indirectP3")
plt.bar(xs + bar_w/2, counts_p5, width=bar_w, alpha=0.85, label="indirectP5")
plt.xlabel("Indirect path count")
plt.ylabel("Number of edges")
plt.title("Histogram of indirectP3 and indirectP5 (values > 0)")
plt.xticks(xs if max_val <= 40 else None)
plt.legend()
plt.grid(True, axis="y", alpha=0.3)
plt.show()

# ---------------- Plot 2: percentage of total edges, side-by-side bars ----------------
pct_p3 = (counts_p3 / n_edges_total) * 100.0
pct_p5 = (counts_p5 / n_edges_total) * 100.0

plt.figure(figsize=(12, 7))
plt.bar(xs - bar_w/2, pct_p3, width=bar_w, alpha=0.85, label="indirectP3")
plt.bar(xs + bar_w/2, pct_p5, width=bar_w, alpha=0.85, label="indirectP5")
plt.xlabel("Indirect path count")
plt.ylabel("Percent of total edges (%)")
plt.title("Indirect path counts as % of all edges in g")
plt.xticks(xs if max_val <= 40 else None)
plt.legend()
plt.grid(True, axis="y", alpha=0.3)
plt.show()

# ---------------- Plot 3: raw counts, side-by-side bars with log Y axis ----------------
plt.figure(figsize=(12, 7))
plt.bar(xs - bar_w/2, counts_p3, width=bar_w, alpha=0.85, label="indirectP3")
plt.bar(xs + bar_w/2, counts_p5, width=bar_w, alpha=0.85, label="indirectP5")
plt.yscale("log")
plt.xlabel("Indirect path count")
plt.ylabel("Number of edges (log scale)")
plt.title("Histogram of indirectP3 and indirectP5 (log Y, values > 0)")
plt.xticks(xs if max_val <= 40 else None)
plt.legend()
plt.grid(True, axis="y", alpha=0.3)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Extract edge-property array ---
if "indirectP3" not in g.ep:
    raise RuntimeError("Graph is missing edge property 'indirectP3'.")

p3 = g.ep["indirectP3"].a
p3_pos = p3[p3 > 0]

n_edges_total = int(g.num_edges())

print(f"Total edges in g: {n_edges_total:,}")
print(f"indirectP3 > 0: {p3_pos.size:,}")

if p3_pos.size == 0:
    raise RuntimeError("No positive values found in indirectP3.")

max_val = int(p3_pos.max())
xs = np.arange(1, max_val + 1)
bins = np.arange(0.5, max_val + 1.5, 1)

counts_p3, _ = np.histogram(p3_pos, bins=bins)
pct_p3 = (counts_p3 / n_edges_total) * 100.0

bar_w = 0.8

# ---------------- 2x2 layout with 3 plots used ----------------
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
ax1, ax2 = axes[0]
ax3, ax4 = axes[1]

# Plot 1: raw counts
ax1.bar(xs, counts_p3, width=bar_w, alpha=0.85, label="indirectP3")
ax1.set_xlabel("Indirect path count")
ax1.set_ylabel("Number of edges")
ax1.set_title("indirectP3 count")
if max_val <= 40:
    ax1.set_xticks(xs)
ax1.legend()
ax1.grid(True, axis="y", alpha=0.3)

# Plot 2: percentage of total edges
ax2.bar(xs, pct_p3, width=bar_w, alpha=0.85, label="indirectP3")
ax2.set_xlabel("Indirect path count")
ax2.set_ylabel("Percent of total edges (%)")
ax2.set_title("indirectP3 as % of all edges")
if max_val <= 40:
    ax2.set_xticks(xs)
ax2.legend()
ax2.grid(True, axis="y", alpha=0.3)

# Plot 3: raw counts with log Y axis
ax3.bar(xs, counts_p3, width=bar_w, alpha=0.85, label="indirectP3")
ax3.set_yscale("log")
ax3.set_xlabel("Indirect path count")
ax3.set_ylabel("Number of edges (log scale)")
ax3.set_title("indirectP3 count (log Y)")
if max_val <= 40:
    ax3.set_xticks(xs)
ax3.legend()
ax3.grid(True, axis="y", alpha=0.3)

# Empty 4th panel
ax4.axis("off")

plt.tight_layout()
plt.show()

## Filter A: keep edges with `indirectP3 >= p3_filter` OR `indirectP5 >= p5_filter`, then save the LCC

In [ ]:
def save_lcc_with_filters(g, p3_filter, p5_filter, out_path: str, weight_ep: str = "weight"):
    p3 = g.ep["indirectP3"].a
    p5 = g.ep["indirectP5"].a

    # 1) Build bool edge filter: support == 1
    efilt = g.new_ep("bool")
    # Fast path for numeric property maps backed by arrays
    efilt.a = (p3 >= p3_filter) | (p5 >= p5_filter)
    print("Number of edges that pass filter=", np.sum(efilt.a))

    gv_filt = GraphView(g, efilt=efilt)

    # 2) Largest connected component on the filtered graph
    lcc_vfilt = label_largest_component(gv_filt, directed=False)
    gv_lcc = GraphView(gv_filt, vfilt=lcc_vfilt)

    # 3) Stats
    n_v = gv_lcc.num_vertices()
    n_e = gv_lcc.num_edges()

    total_w = None
    if weight_ep is not None and weight_ep in g.ep:
        w = g.ep[weight_ep]
        total_w = float(sum(w[e] for e in gv_lcc.edges()))
        print(f"[p3_filter={p3_filter}, p5_filter={p5_filter}, LCC] |V|={n_v:,}  |E|={n_e:,}  total_{weight_ep}={total_w:,.3f}")
    else:
        if weight_ep is not None:
            print(f"Note: edge property '{weight_ep}' not found; skipping total weight.")
        print(f"[p3_filter={p3_filter}, p5_filter={p5_filter}, LCC] |V|={n_v:,}  |E|={n_e:,}")

    # 4) Save
    g_to_save = gt.Graph(gv_lcc, prune=True)
    g_to_save.save(out_path)
    print(f"Saved to {out_path}")

    return g_to_save

In [ ]:
p3_filter = 1
p5_filter = 1
out_path = f"{scratch_prefix}{SAMPLE_ID}_barcodesGraph_{suffix}_filter_{p3_filter}_{p5_filter}.gt.gz"

g2 = save_lcc_with_filters(
    g,
    p3_filter,
    p5_filter,
    out_path=out_path
)

## Filter B: keep edges with `indirectP3 + indirectP5 >= sum_filter`, then save the LCC

In [ ]:
def save_lcc_with_sum_filters(g, sum_filter, out_path: str, weight_ep: str = "weight"):
    p3 = g.ep["indirectP3"].a
    p5 = g.ep["indirectP5"].a

    # 1) Build bool edge filter: support == 1
    efilt = g.new_ep("bool")
    # Fast path for numeric property maps backed by arrays
    efilt.a = (p3 + p5 >= sum_filter)
    print("Number of edges that pass filter=", np.sum(efilt.a))

    gv_filt = GraphView(g, efilt=efilt)

    # 2) Largest connected component on the filtered graph
    lcc_vfilt = label_largest_component(gv_filt, directed=False)
    gv_lcc = GraphView(gv_filt, vfilt=lcc_vfilt)

    # 3) Stats
    n_v = gv_lcc.num_vertices()
    n_e = gv_lcc.num_edges()

    total_w = None
    if weight_ep is not None and weight_ep in g.ep:
        w = g.ep[weight_ep]
        total_w = float(sum(w[e] for e in gv_lcc.edges()))
        print(f"[sumFilter={sum_filter}, LCC] |V|={n_v:,}  |E|={n_e:,}  total_{weight_ep}={total_w:,.3f}")
    else:
        if weight_ep is not None:
            print(f"Note: edge property '{weight_ep}' not found; skipping total weight.")
        print(f"[sumFilter={sum_filter}, LCC] |V|={n_v:,}  |E|={n_e:,}")

    # 4) Save
    g_to_save = gt.Graph(gv_lcc, prune=True)
    g_to_save.save(out_path)
    print(f"Saved to {out_path}")

    return g_to_save

In [ ]:
sum_filter = 2
out_path = f"{scratch_prefix}{SAMPLE_ID}_barcodesGraph_{suffix}_sumFilter_{sum_filter}.gt.gz"

g2 = save_lcc_with_sum_filters(
    g,
    sum_filter,
    out_path=out_path
)